In [1]:
# team 121
# https://pytorch-forecasting.readthedocs.io/en/latest/tutorials/stallion.html
!pip install pytorch-forecasting --extra-index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu


In [2]:
import numpy as np
import pandas as pd
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.models import TemporalFusionTransformer
from pytorch_forecasting.metrics.point import MAE
from pytorch_forecasting.utils import move_to_device
from lightning.pytorch import Trainer

In [3]:
class ZillowHousingDataService:
    
    def __init__(self, new_construction_price_per_square_foot_data_set_empty={}):
        self.new_construction_price_per_square_foot_data_set_formatted = new_construction_price_per_square_foot_data_set_empty
        
    def get_new_construction_price_per_square_foot_data_set_formatted(self):
        return self.new_construction_price_per_square_foot_data_set_formatted
        
    def format_new_construction_price_per_square_foot_data_set(self, create_repeating_array_function, data_frames, region_ids=None):
        
        if region_ids == None:
            index_values = data_frames.index
            
        else:
            index_values = region_ids
            
        for index_value in index_values:
            
            new_data_frame = data_frames.loc[[index_value]]
            
            time_series_data = new_data_frame.get(data_frames.columns[4:len(data_frames.columns)])
            
            new_data_frame.drop(columns=new_data_frame.columns, inplace=True)
            
            new_data_frame = new_data_frame.assign(RegionID=[create_repeating_array_function(index_value, len(time_series_data.columns))])
            
            new_data_frame = new_data_frame.explode("RegionID")
            
            new_data_frame.reset_index(drop=True, inplace=True)
            
            new_data_frame.insert(loc=0, column="Index", value=new_data_frame.index, allow_duplicates=False)
            
            new_data_frame = new_data_frame.assign(Date=time_series_data.columns.T)
            
            new_data_frame = new_data_frame.assign(Price=time_series_data.T.to_numpy(copy=True))
            
            self.new_construction_price_per_square_foot_data_set_formatted[index_value] = new_data_frame

In [4]:
data_file_path = "~/ece492-final-project/"
data_file_name = "Metro_new_con_median_sale_price_per_sqft_uc_sfrcondo_month.csv"

data_frames = pd.read_csv(data_file_path + data_file_name)
data_frames.set_index("RegionID", inplace=True)
data_frames.sort_index(inplace=True)

In [5]:
data_service = ZillowHousingDataService()
data_service.format_new_construction_price_per_square_foot_data_set(np.repeat, data_frames)
formatted_data_sets = data_service.get_new_construction_price_per_square_foot_data_set_formatted()

In [6]:
# Raleigh, NC - 395012
data = formatted_data_sets[395012]

In [7]:
training = TimeSeriesDataSet(
    data=data,
    time_idx="Index",
    target="Price",
    group_ids=["RegionID"]
)

In [8]:
batch_size = 128
validation = TimeSeriesDataSet.from_dataset(training, data, min_prediction_idx=training.index.time.max() + 1, stop_randomization=True)
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=31)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size, num_workers=31)

In [9]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.01,  # lower = more stable
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=MAE(),
)

trainer = Trainer(
    max_epochs=30,
    accelerator="cuda",
)

trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

raw_predictions = tft.predict(
    val_dataloader,
    mode="raw",
    return_x=True,
    trainer_kwargs=dict(accelerator="cuda")
)

predictions = tft.predict(
    val_dataloader,
    return_x=True,
    trainer_kwargs=dict(accelerator="cuda")
)

predictions_vs_actuals = tft.calculate_prediction_actual_by_variable(
    predictions.x,
    predictions.output
)

#tft.plot_prediction_actual_by_variable(predictions_vs_actuals)

/opt/conda/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/conda/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/opt/conda/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `t

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/opt/conda/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/conda/lib/python3.13/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=30` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://